# SONA VoiceBank GTCRN Capacity/Loss Sweep

Layer B sweep: trains multiple Bark-band `gtcrn` capacities/loss weights on VoiceBank, evaluates each frozen model plus the locked strict guarded online variant, and stores checkpoints/results in one zip.


In [ ]:
%pip install -q pesq pystoi ptflops

import torch
import torchaudio

print('torch', torch.__version__)
print('torchaudio', torchaudio.__version__)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on GPU accelerator before running.'

In [ ]:
import glob
from pathlib import Path

def find_repo():
    hits = glob.glob('/kaggle/input/**/scripts/kaggle_tune_online.py', recursive=True)
    hits = sorted(hits, key=lambda p: ('sona' not in p.lower(), len(p)))
    return str(Path(hits[0]).parents[1]) if hits else None

def has_vbd_layout(root):
    root = Path(root)
    needed = ['clean_trainset_wav', 'noisy_trainset_wav', 'clean_testset_wav', 'noisy_testset_wav']
    return all((root / name).is_dir() for name in needed)

def find_vbd():
    hits = glob.glob('/kaggle/input/**/noisy_trainset_wav', recursive=True)
    roots = [str(Path(h).parent) for h in hits if has_vbd_layout(Path(h).parent)]
    return sorted(roots, key=len)[0] if roots else None

REPO_DIR = find_repo()
VBD_ROOT = find_vbd()
OUT_DIR = '/kaggle/working/results_gtcrn_sweep'

# If needed, set manually from the Kaggle right panel.
# REPO_DIR = '/kaggle/input/sona-repository'
# VBD_ROOT = '/kaggle/input/voicebank-demand-16k'

assert REPO_DIR, 'Updated sona repository not found.'
assert VBD_ROOT, 'VoiceBank-DEMAND folders not found.'
assert (Path(REPO_DIR) / 'src' / 'models' / 'gtcrn.py').exists(), 'sona repository is old. Upload the refreshed sona.zip first.'

print('REPO_DIR =', REPO_DIR)
print('VBD_ROOT =', VBD_ROOT)
print('OUT_DIR  =', OUT_DIR)

In [ ]:
# Balanced sweep. Expected Kaggle T4 time: roughly 5-8 hours.
# Increase LIMIT_TEST to 824 only after a config wins this 300-clip sweep.
LIMIT_TRAIN = 2000
LIMIT_TEST = 300
EPOCHS = 50

SUP_MODELS = 'gtcrn'
GTCRN_CHS = '64,96'
GTCRN_HIDDENS = '96,128'
GTCRN_BLOCKS = '4,5'
MAG_WEIGHTS = '0.5,0.75'

# Keep online path strict and minimal; this run selects the backbone first.
ADAPTERS = 'lowrank'
ANCHORS = '2.0'
STEPS = '1'
MAX_GAIN_DELTAS = '0.01'
GUARD_MIN_REWARD_DELTAS = 'none'
GUARD_MAX_OUTPUT_DELTAS = '0.0001'
GUARD_MIN_MASK_POLARITIES = 'none'
MASK_POLARITY_WEIGHT = 0.0
MASK_POLARITY_TARGET = 0.0
SMOOTH_TIME_WEIGHT = 0.0
SMOOTH_BAND_WEIGHT = 0.0
LR = 0.1

print(dict(
    LIMIT_TRAIN=LIMIT_TRAIN,
    LIMIT_TEST=LIMIT_TEST,
    EPOCHS=EPOCHS,
    SUP_MODELS=SUP_MODELS,
    GTCRN_CHS=GTCRN_CHS,
    GTCRN_HIDDENS=GTCRN_HIDDENS,
    GTCRN_BLOCKS=GTCRN_BLOCKS,
    MAG_WEIGHTS=MAG_WEIGHTS,
    ADAPTERS=ADAPTERS,
    ANCHORS=ANCHORS,
    STEPS=STEPS,
    MAX_GAIN_DELTAS=MAX_GAIN_DELTAS,
    GUARD_MAX_OUTPUT_DELTAS=GUARD_MAX_OUTPUT_DELTAS,
))

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, f'{REPO_DIR}/scripts/kaggle_tune_online.py',
    '--repo', REPO_DIR,
    '--vbd-root', VBD_ROOT,
    '--out', OUT_DIR,
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-test', str(LIMIT_TEST),
    '--epochs', str(EPOCHS),
    '--sup-models', SUP_MODELS,
    '--gtcrn-chs', GTCRN_CHS,
    '--gtcrn-hiddens', GTCRN_HIDDENS,
    '--gtcrn-blocks', GTCRN_BLOCKS,
    '--mag-weights', MAG_WEIGHTS,
    '--adapters', ADAPTERS,
    '--anchors', ANCHORS,
    '--steps', STEPS,
    '--max-gain-deltas', MAX_GAIN_DELTAS,
    '--guard-min-reward-deltas', GUARD_MIN_REWARD_DELTAS,
    '--guard-max-output-deltas', GUARD_MAX_OUTPUT_DELTAS,
    '--guard-min-mask-polarities', GUARD_MIN_MASK_POLARITIES,
    '--mask-polarity-weight', str(MASK_POLARITY_WEIGHT),
    '--mask-polarity-target', str(MASK_POLARITY_TARGET),
    '--smooth-time-weight', str(SMOOTH_TIME_WEIGHT),
    '--smooth-band-weight', str(SMOOTH_BAND_WEIGHT),
    '--lr', str(LR),
]

print(' '.join(cmd))
print()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
assert proc.returncode == 0, f'kaggle_tune_online.py exited {proc.returncode}'

In [ ]:
import pandas as pd
from pathlib import Path

summary_md = Path(OUT_DIR) / 'summary.md'
summary_csv = Path(OUT_DIR) / 'summary.csv'
print(summary_md.read_text(encoding='utf-8'))
df = pd.read_csv(summary_csv)
display(df.sort_values(['pesq', 'si_sdr_i'], ascending=[False, False]))

frozen = df[df.method.str.startswith('neural_gtcrn')].copy()
display(frozen.sort_values(['pesq', 'si_sdr_i', 'estoi'], ascending=[False, False, False]))
display(frozen.sort_values(['si_sdr_i', 'pesq', 'estoi'], ascending=[False, False, False]))
display(frozen.sort_values(['estoi', 'stoi', 'pesq'], ascending=[False, False, False]))

In [ ]:
import glob
import os
import shutil
from IPython.display import FileLink, display

ART = '/kaggle/working/sona_voicebank_gtcrn_sweep'
shutil.make_archive(ART, 'zip', OUT_DIR)
print(f'Wrote {ART}.zip ({os.path.getsize(ART + ".zip") / 1e6:.1f} MB)')
for f in sorted(glob.glob(OUT_DIR + '/**', recursive=True)):
    if os.path.isfile(f):
        print('  ', os.path.relpath(f, OUT_DIR), f'({os.path.getsize(f) / 1e3:.0f} KB)')
display(FileLink(ART + '.zip'))